<a href="https://colab.research.google.com/github/mancinigabriel/tcc-pece-assin2-llm-challenges/blob/main/notebooks/Ministral3_Reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Preparando ambiente

In [1]:
!git clone https://github.com/mancinigabriel/tcc-pece-assin2-llm-challenges.git

Cloning into 'tcc-pece-assin2-llm-challenges'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 50 (delta 19), reused 34 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 96.86 KiB | 8.80 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [2]:
!pip install transformers==5.0.0rc0
!pip install mistral-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 139.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.4/533.4 kB 50.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 149.3 MB/s eta 0:00:00


In [3]:
import sys
import os

PROJECT_ROOT = os.path.abspath('/content/tcc-pece-assin2-llm-challenges')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend, FineGrainedFP8Config
from datetime import datetime, timezone, timedelta
from src import prompts
from src import utils
from src import data
from src import metrics
import pandas as pd
import random
import torch
import re
import time

In [6]:
cfg = utils.load_config(
    "/content/tcc-pece-assin2-llm-challenges/configs/base.yaml",
    "/content/tcc-pece-assin2-llm-challenges/configs/models/mistral3.yaml"
)

generation_args = cfg["generation"]

gpu = 'A100 RAM alta'

#Ministral 3 - 3B - Reasoning

In [7]:
timing = {}

timing['inicio'] = utils.time_log()


model_id = "mistralai/Ministral-3-3B-Reasoning-2512"
model_name = "mistral3b_reasoning"
quantizado = False
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
)
tokenizer = MistralCommonBackend.from_pretrained(model_id)

df_assin_2 = data.gera_df()

config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

tekken.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

##Teste de Consistência

In [8]:
timing['inicio_cons'] = utils.time_log()

df_assin_2_consistency_test = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_consistency_test)):
    premissa = df_assin_2_consistency_test.iloc[i]['premise']
    hipotese = df_assin_2_consistency_test.iloc[i]['hypothesis']

    prompt = prompts.zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, **generation_args)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'test_{j}'

    df_assin_2_consistency_test.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))

df_assin_2_consistency_test.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_consistencia.csv')

timing['fim_cons'] = utils.time_log()

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-11 23:04:58
100 - 2026-01-11 23:05:13
200 - 2026-01-11 23:05:29
300 - 2026-01-11 23:05:45
400 - 2026-01-11 23:06:00


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-11 23:06:16
100 - 2026-01-11 23:06:32
200 - 2026-01-11 23:06:47
300 - 2026-01-11 23:07:03
400 - 2026-01-11 23:07:19


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


...

/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-11 23:08:53
100 - 2026-01-11 23:09:08
200 - 2026-01-11 23:09:24
300 - 2026-01-11 23:09:39
400 - 2026-01-11 23:09:55


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-11 23:10:11
100 - 2026-01-11 23:10:26
200 - 2026-01-11 23:10:42
300 - 2026-01-11 23:10:57
400 - 2026-01-11 23:11:13


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))


##Loop de aplicação do prompt em todo o dataset

In [9]:
timing['inicio_aplicacao_total'] = utils.time_log()

for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2[column_name]))
df_assin_2.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}.csv')

timing['fim'] = utils.time_log()

metrics_dict = {'acurácia': metrics.calculate_accuracy(df_assin_2),
           'consistência': metrics.compute_consistency(df_assin_2_consistency_test)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, model_name)

0 - 2026-01-11 23:11:32
100 - 2026-01-11 23:11:48
200 - 2026-01-11 23:12:03
300 - 2026-01-11 23:12:19
400 - 2026-01-11 23:12:34
500 - 2026-01-11 23:12:50
600 - 2026-01-11 23:13:05
700 - 2026-01-11 23:13:21
800 - 2026-01-11 23:13:36
900 - 2026-01-11 23:13:52
1000 - 2026-01-11 23:14:07
1100 - 2026-01-11 23:14:23
1200 - 2026-01-11 23:14:38
1300 - 2026-01-11 23:14:54
1400 - 2026-01-11 23:15:09
1500 - 2026-01-11 23:15:25
1600 - 2026-01-11 23:15:40
1700 - 2026-01-11 23:15:56
1800 - 2026-01-11 23:16:11
1900 - 2026-01-11 23:16:27
2000 - 2026-01-11 23:16:43
2100 - 2026-01-11 23:16:58
2200 - 2026-01-11 23:17:14
2300 - 2026-01-11 23:17:30
2400 - 2026-01-11 23:17:45
2500 - 2026-01-11 23:18:01
2600 - 2026-01-11 23:18:17
2700 - 2026-01-11 23:18:32
2800 - 2026-01-11 23:18:48
2900 - 2026-01-11 23:19:04
3000 - 2026-01-11 23:19:19
3100 - 2026-01-11 23:19:35
3200 - 2026-01-11 23:19:51
3300 - 2026-01-11 23:20:06
3400 - 2026-01-11 23:20:21
3500 - 2026-01-11 23:20:37
3600 - 2026-01-11 23:20:52
3700 - 2026-0

'Arquivo salvo em /content/drive/MyDrive/Mestrado/TCC Pós/Dados/logs/log_mistral3b_reasoning_20260111_233536.json às 2026-01-11 23:35:36'

# Mistral 3 - 8B - Reasoning

In [10]:
timing = {}

timing['inicio'] = utils.time_log()

model_id = "mistralai/Ministral-3-8B-Reasoning-2512"
model_name = "mistral8b_reasoning"
quantizado = False
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
)
tokenizer = MistralCommonBackend.from_pretrained(model_id)

df_assin_2 = data.gera_df()

config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'max_position_embeddings'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

tekken.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

In [11]:
timing['inicio_cons'] = utils.time_log()

df_assin_2_consistency_test = df_assin_2.head(500)

for j in range(5):
  for i in range(len(df_assin_2_consistency_test)):
    premissa = df_assin_2_consistency_test.iloc[i]['premise']
    hipotese = df_assin_2_consistency_test.iloc[i]['hypothesis']

    prompt = prompts.zero_shot_prompt(premissa, hipotese)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, **generation_args)
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][prompt_len:]
    resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    column_name = f'test_{j}'

    df_assin_2_consistency_test.loc[i, column_name] = resp

    if i%100==0:
      print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))

df_assin_2_consistency_test.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}_consistencia.csv')

timing['fim_cons'] = utils.time_log()

/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-11 23:36:47
100 - 2026-01-11 23:37:17
200 - 2026-01-11 23:37:47
300 - 2026-01-11 23:38:17
400 - 2026-01-11 23:38:47


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-11 23:39:17
100 - 2026-01-11 23:39:47
200 - 2026-01-11 23:40:17
300 - 2026-01-11 23:40:47
400 - 2026-01-11 23:41:17


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


...

/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-11 23:44:17
100 - 2026-01-11 23:44:48
200 - 2026-01-11 23:45:18
300 - 2026-01-11 23:45:48
400 - 2026-01-11 23:46:18


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))
/tmp/ipython-input-3760482454.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test.loc[i, column_name] = resp


0 - 2026-01-11 23:46:48
100 - 2026-01-11 23:47:18
200 - 2026-01-11 23:47:48
300 - 2026-01-11 23:48:18
400 - 2026-01-11 23:48:48


/tmp/ipython-input-3760482454.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_assin_2_consistency_test[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2_consistency_test[column_name]))


In [12]:
timing['inicio_aplicacao_total'] = utils.time_log()

for i in range(len(df_assin_2)):
  premissa = df_assin_2.iloc[i]['premise']
  hipotese = df_assin_2.iloc[i]['hypothesis']

  prompt = prompts.zero_shot_prompt(premissa, hipotese)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(**inputs, **generation_args)
  prompt_len = inputs["input_ids"].shape[1]
  generated_tokens = output[0][prompt_len:]
  resp = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  column_name = f'pred'

  df_assin_2.loc[i, column_name] = resp

  if i%100==0:
    print(f"{i} - {utils.time_log().strftime("%Y-%m-%d %H:%M:%S")}")

df_assin_2[f'{column_name}_tratado'] = list(map(lambda x: utils.extract_response_character(x), df_assin_2[column_name]))
df_assin_2.to_csv(f'/content/drive/MyDrive/Mestrado/TCC Pós/Dados/dados_{model_name}.csv')

timing['fim'] = utils.time_log()

metrics_dict = {'acurácia': metrics.calculate_accuracy(df_assin_2),
           'consistência': metrics.compute_consistency(df_assin_2_consistency_test)}

log = utils.log(model_id, gpu, quantizado, generation_args, metrics_dict, timing)
utils.export_log(log, model_name)

0 - 2026-01-11 23:49:18
100 - 2026-01-11 23:49:48
200 - 2026-01-11 23:50:18
300 - 2026-01-11 23:50:48
400 - 2026-01-11 23:51:18
500 - 2026-01-11 23:51:48
600 - 2026-01-11 23:52:18
700 - 2026-01-11 23:52:48
800 - 2026-01-11 23:53:18
900 - 2026-01-11 23:53:49
1000 - 2026-01-11 23:54:18
1100 - 2026-01-11 23:54:48
1200 - 2026-01-11 23:55:18
1300 - 2026-01-11 23:55:48
1400 - 2026-01-11 23:56:18
1500 - 2026-01-11 23:56:48
1600 - 2026-01-11 23:57:19
1700 - 2026-01-11 23:57:49
1800 - 2026-01-11 23:58:19
1900 - 2026-01-11 23:58:49
2000 - 2026-01-11 23:59:19
2100 - 2026-01-11 23:59:49
2200 - 2026-01-12 00:00:19
2300 - 2026-01-12 00:00:50
2400 - 2026-01-12 00:01:20
2500 - 2026-01-12 00:01:50
2600 - 2026-01-12 00:02:20
2700 - 2026-01-12 00:02:50
2800 - 2026-01-12 00:03:21
2900 - 2026-01-12 00:03:51
3000 - 2026-01-12 00:04:21
3100 - 2026-01-12 00:04:52
3200 - 2026-01-12 00:05:22
3300 - 2026-01-12 00:05:52
3400 - 2026-01-12 00:06:21
3500 - 2026-01-12 00:06:51
3600 - 2026-01-12 00:07:20
3700 - 2026-0

'Arquivo salvo em /content/drive/MyDrive/Mestrado/TCC Pós/Dados/logs/log_mistral8b_reasoning_20260112_003544.json às 2026-01-12 00:35:44'